## Introduction

Attention heads in Vision Transformers are often redundant — Michel et al. (2019) showed that many heads can be removed with minimal accuracy loss. **Head pruning** removes entire attention heads from a transformer, reducing both parameters and computation.

fasterai's `Pruner` leverages [torch-pruning](https://github.com/VainF/Torch-Pruning)'s built-in head pruning support. When head pruning is enabled, the Pruner automatically patches timm attention modules to be pruning-compatible (following the official [torch-pruning ViT examples](https://github.com/VainF/Torch-Pruning/tree/master/examples/transformers)).

## Setup

In [1]:
import torch, torch.nn as nn
from fasterai.prune.pruner import Pruner
from fasterai.core.criteria import large_final

## Quick Example

Prune 50% of attention heads from a timm ViT — just add `head_pruning_ratio`:

In [2]:
from timm import create_model

model = create_model('vit_small_patch16_224', pretrained=True, num_classes=10)
x = torch.randn(1, 3, 224, 224)
params_before = sum(p.numel() for p in model.parameters())

pruner = Pruner(model, pruning_ratio=0.3, context='local', criteria=large_final,
                example_inputs=x, head_pruning_ratio=0.5)
pruner.prune_model()

attn = model.blocks[0].attn
params_after = sum(p.numel() for p in model.parameters())
print(f'Before: 6 heads, embed_dim=384, params={params_before:,}')
print(f'After:  {attn.num_heads} heads, embed_dim={attn.num_heads * attn.head_dim}, params={params_after:,} ({100*(1-params_after/params_before):.0f}% reduction)')
print()
print(f'Inference OK: {model(x).shape}')

Ignoring output layer: head
Detected 12 attention layer(s) (will be pruned)
Total ignored layers: 1


/home/nathan/miniconda3/envs/dev/lib/python3.12/site-packages/torch_pruning/dependency.py:712: UserWarning: Unwrapped parameters detected: ['cls_token', 'pos_embed'].
 Torch-Pruning will prune the last non-singleton dimension of these parameters. If you wish to change this behavior, please provide an unwrapped_parameters argument.
  warnings.warn(warning_str)


Before: 6 heads, embed_dim=384, params=21,669,514
After:  3 heads, embed_dim=192, params=9,685,778 (55% reduction)

Inference OK: torch.Size([1, 10])


`pruning_ratio` and `head_pruning_ratio` are independent — you can use different values. The Pruner automatically patches the attention forward method to handle the dimension changes.

## Understanding the Parameters

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `head_pruning_ratio` | float | 0.0 | Ratio of attention heads to remove (0–1 or 0–100) |
| `prune_num_heads` | bool | False | Remove entire attention heads |
| `prune_head_dims` | bool | True | Reduce head dimensions instead |

### Auto-enable XOR pattern

Setting `head_pruning_ratio > 0` automatically enables `prune_num_heads=True` and sets `prune_head_dims=False`. This follows torch-pruning's convention: **remove whole heads OR reduce dimensions, not both**.

Values > 1 are treated as percentages: `head_pruning_ratio=50` becomes `0.5`.

## Channel vs Head vs Head-Dim Pruning

| Approach | What changes | Effect on transformer | Best for |
|----------|-------------|----------------------|----------|
| **Channel pruning** (`pruning_ratio` only) | Reduces `embed_dim` uniformly | All heads get smaller | CNNs, FFN layers |
| **Head pruning** (`head_pruning_ratio`) | Removes entire heads | Fewer heads, same `head_dim` | Transformers with redundant heads |
| **Head dim reduction** (`prune_head_dims=True`) | Shrinks each head | Same heads, smaller `head_dim` | Fine-grained transformer compression |

## Prune-then-Fine-tune Workflow

The recommended workflow: **prune once with `Pruner`, then fine-tune the pruned model**.

In [3]:
from timm import create_model
from fastai.vision.all import *

# Step 1: Load data
path = untar_data(URLs.CIFAR)
dls = ImageDataLoaders.from_folder(path, valid='test', bs=32,
                                   item_tfms=Resize(224))

In [4]:
# Step 2: Prune standalone
model = create_model('vit_small_patch16_224', pretrained=True, num_classes=10)
pruner = Pruner(model, pruning_ratio=0.3, context='local', criteria=large_final,
                example_inputs=torch.randn(1, 3, 224, 224), head_pruning_ratio=0.5)
pruner.prune_model()
print(f'Pruned: 6 -> {model.blocks[0].attn.num_heads} heads per block')

# Step 3: Fine-tune the pruned model
learn = Learner(dls, model, metrics=accuracy, loss_func=CrossEntropyLossFlat())
learn.fit(5, lr=1e-4)

Ignoring output layer: head
Detected 12 attention layer(s) (will be pruned)
Total ignored layers: 1


/home/nathan/miniconda3/envs/dev/lib/python3.12/site-packages/torch_pruning/dependency.py:712: UserWarning: Unwrapped parameters detected: ['cls_token', 'pos_embed'].
 Torch-Pruning will prune the last non-singleton dimension of these parameters. If you wish to change this behavior, please provide an unwrapped_parameters argument.
  warnings.warn(warning_str)


Pruned: 6 -> 3 heads per block


epoch,train_loss,valid_loss,accuracy,time
0,1.028984,1.145234,0.587600,00:39
1,0.817868,0.824215,0.701300,00:39
2,0.709084,0.745389,0.735800,00:39
3,0.609338,0.697566,0.757400,00:39
4,0.507757,0.662502,0.773400,00:39


## Pruning Report

Use `print_sparsity()` to see head count changes alongside parameter reduction:

## Training with PruneCallback

Head pruning also works with `PruneCallback` for training-time pruning. Heads are removed once early in training, then channel pruning continues gradually. The Pruner automatically freezes head pruning after the first application to prevent over-pruning.

In [5]:
from fasterai.prune.prune_callback import PruneCallback
from fasterai.core.schedule import one_shot, agp

# Load data
path = untar_data(URLs.CIFAR)
dls = ImageDataLoaders.from_folder(path, valid='test', bs=32,
                                   item_tfms=Resize(224))

In [6]:
# Load model
model = create_model('vit_small_patch16_224', pretrained=True, num_classes=10)

# PruneCallback with head pruning — heads removed once, channels pruned gradually
cb = PruneCallback(
    pruning_ratio=0.1,          # 30% channel pruning (gradual via AGP)
    schedule=agp,
    context='local',
    criteria=large_final,
    head_pruning_ratio=0.1,    # 50% head removal (applied once, then frozen)
)

learn = Learner(dls, model, metrics=accuracy, loss_func=CrossEntropyLossFlat())
learn.fit(3, lr=1e-3, cbs=[cb])

Ignoring output layer: head
Detected 12 attention layer(s) (will be pruned)
Total ignored layers: 1


/home/nathan/miniconda3/envs/dev/lib/python3.12/site-packages/torch_pruning/dependency.py:712: UserWarning: Unwrapped parameters detected: ['cls_token', 'pos_embed'].
 Torch-Pruning will prune the last non-singleton dimension of these parameters. If you wish to change this behavior, please provide an unwrapped_parameters argument.
  warnings.warn(warning_str)


epoch,train_loss,valid_loss,accuracy,time
0,2.447181,2.497135,0.117900,04:15


Sparsity at the end of epoch 0: 7.03%


KeyboardInterrupt: 

In [7]:
# Inspect results
print(f'Heads: 6 -> {model.blocks[0].attn.num_heads}')
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')
print(f'Inference: {model(torch.randn(1, 3, 224, 224)).shape}')

Heads: 6 -> 5
Params: 17,868,352


RuntimeError: Input type (torch.FloatTensor) and weight type (torch.cuda.FloatTensor) should be the same or input should be a MKLDNN tensor and weight is a dense tensor

In [8]:
pruner.print_sparsity()


Pruning Report:
-------------------------------------------------------------------------------------
Layer                               Type         In Ch    Out Ch   Params      
-------------------------------------------------------------------------------------
patch_embed.proj                    Conv2d       3        268      206,092     
blocks.0.attn.qkv                   Linear       268      576      154,944     
blocks.0.attn.proj                  Linear       192      268      51,724      
blocks.0.mlp.fc1                    Linear       268      1075     289,175     
blocks.0.mlp.fc2                    Linear       1075     268      288,368     
blocks.1.attn.qkv                   Linear       268      576      154,944     
blocks.1.attn.proj                  Linear       192      268      51,724      
blocks.1.mlp.fc1                    Linear       268      1075     289,175     
blocks.1.mlp.fc2                    Linear       1075     268      288,368     
blocks.2.at

## Supported Architectures

Head pruning works with attention modules that have a fused QKV Linear layer (`.qkv` attribute). The Pruner auto-patches their forward method for pruning compatibility.

| Architecture | Supported | Notes |
|-------------|-----------|-------|
| **timm ViT, DeiT, Swin** | **Yes** | Auto-detected and patched |
| `nn.MultiheadAttention` | No | Uses raw `in_proj_weight`, not Linear submodules |
| HuggingFace ViT | Not yet | Separate Q/K/V projections (planned) |

---

## Summary

| Tool / Function | Purpose |
|----------------|----------|
| `Pruner(..., head_pruning_ratio=0.5)` | Remove 50% of attention heads |
| `PruneCallback(..., head_pruning_ratio=0.5)` | Head pruning during training (heads frozen after first step) |
| `prune_num_heads / prune_head_dims` | XOR: remove heads vs reduce dim |
| `print_sparsity()` | Report with head count changes |

---

## See Also

- [Pruner](../prune/pruner.html) — Full Pruner API reference
- [PruneCallback](../prune/prune_callback.html) — Structured pruning during training (CNNs)
- [Criteria](../core/criteria.html) — Importance measures for selecting what to prune
- [Transformer Sparsification](transformers.html) — Unstructured sparsification alternative

---

## Prototype: `prune_every` for faster training

PruneCallback calls `prune_model()` every batch (~154ms overhead). This prototype keeps the per-batch schedule (smooth AGP curve) but only fires the expensive graph traversal + pruning every N batches. On skipped batches, the schedule counter still advances — so when pruning does fire, it catches up to the correct cumulative ratio.

In [15]:
from fasterai.prune.prune_callback import PruneCallback

class PruneEveryCallback(PruneCallback):
    "PruneCallback that only fires pruning every N batches (schedule stays per-batch)"
    def __init__(self, pruning_ratio, schedule, context, criteria,
                 prune_every=1,  # int (every N batches) or 'epoch'
                 *args, **kwargs):
        super().__init__(pruning_ratio, schedule, context, criteria, *args, **kwargs)
        self._prune_every = prune_every

    def before_fit(self):
        "Setup pruner — same as PruneCallback but resolve prune_every='epoch'"
        super().before_fit()
        n_batches = len(self.learn.dls.train)
        self._interval = n_batches if self._prune_every == 'epoch' else self._prune_every
        self._step_count = 0
    
    def before_step(self):
        "Advance schedule every batch, but only prune every N batches"
        if self.training:
            self._step_count += 1
            if self._step_count % self._interval == 0:
                # Catch up: fire the actual pruning (uses current_step from schedule)
                self.pruner.prune_model()
            else:
                # Skip the expensive pruning, just advance the schedule counter
                self.pruner.pruner.current_step += 1

In [17]:
import time
from fasterai.core.schedule import agp

model = create_model('vit_small_patch16_224', pretrained=True, num_classes=10)
cb = PruneEveryCallback(
    pruning_ratio=0.3, schedule=agp, context='local', criteria=large_final,
    head_pruning_ratio=0.5, prune_every='epoch',
)
learn = Learner(dls, model, metrics=accuracy, loss_func=CrossEntropyLossFlat())

t0 = time.perf_counter()
learn.fit(5, lr=1e-3, cbs=[cb])
elapsed = time.perf_counter() - t0

heads = model.blocks[0].attn.num_heads
params = sum(p.numel() for p in model.parameters())
print(f'\nprune_every=epoch | {elapsed:.1f}s | {heads} heads | {params:,} params')

Ignoring output layer: head
Detected 12 attention layer(s) (will be pruned)
Total ignored layers: 1


/home/nathan/miniconda3/envs/dev/lib/python3.12/site-packages/torch_pruning/dependency.py:712: UserWarning: Unwrapped parameters detected: ['cls_token', 'pos_embed'].
 Torch-Pruning will prune the last non-singleton dimension of these parameters. If you wish to change this behavior, please provide an unwrapped_parameters argument.
  warnings.warn(warning_str)


epoch,train_loss,valid_loss,accuracy,time
0,1.462386,1.535220,0.445000,00:53
1,1.499982,1.651546,0.431300,00:47
2,1.652026,1.678159,0.446500,00:44
3,1.696789,1.709670,0.448900,00:41
4,1.698159,1.712002,0.443000,00:42


Sparsity at the end of epoch 0: 14.63%
Sparsity at the end of epoch 1: 23.52%
Sparsity at the end of epoch 2: 28.08%
Sparsity at the end of epoch 3: 29.76%
Sparsity at the end of epoch 4: 30.00%

prune_every=epoch | 228.8s | 4 heads | 10,511,378 params
